In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# How much solid dissolves, and what the activity coefficient is worth (Illustrations 12.1-1 and 12.1-2)

Naphthalene is a solid at 20 °C. Drop it into liquid $n$-hexane and some of it
dissolves. How much?

Section 12.1 answers that with one equation and then spends the rest of the section on
the one quantity in it that is hard to get. The equation is the equality of fugacities,
which for a pure solid in equilibrium with a liquid solution becomes Eq. 12.1-8,

$$\ln x_1 = -\ln\gamma_1 - \frac{\Delta_{\rm fus}H}{RT}\left(1-\frac{T}{T_t}\right)$$

Everything on the right except $\gamma_1$ is a pure-component property of the solid: its
heat of fusion and its melting point. So the **ideal** solubility, $\gamma_1 = 1$, costs
nothing. The rest of the work is $\gamma_1$.

**These two illustrations are the same calculation done three ways**, and they disagree
by a factor of three, which is the point:

| | $\gamma_1$ from | $x_1$ |
|---|---|---|
| Illustration 12.1-1, first | nothing -- ideal solution | 0.269 |
| Illustration 12.1-1, then | regular solution theory | 0.0772 |
| Illustration 12.1-2 | UNIFAC | 0.0856 |
| measured | -- | 0.09 |

The ideal answer is wrong by a factor of three. Regular solution theory gets within
15 %, and UNIFAC within 5 %. That ordering is the argument for Chapter 9's machinery,
made on a single number.

**And there is a loop in it.** $\gamma_1$ depends on composition, and composition is
what is being solved for, so Eq. 12.1-8 cannot be evaluated -- it has to be iterated.
That is exactly what the highlighted sentence in Illustration 12.1-2 says: *since the
output of the model is the activity coefficient, the equation is rewritten* to give
$x_1$ from $\gamma_1$ and gone round again. Both illustrations print their iteration
sequences, and so does this notebook.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.append("..")
import numpy as np

from thermo import sle
from thermo.activity_models import RegularSolution, CAL_CC_HALF, CC_PER_MOL

R = 8.314
T = 20.0 + 273.15                      # 20 C, the temperature of both illustrations

# --- Illustration 12.1-1's data, verbatim ----------------------------------
MW_N     = 128.19                      # naphthalene
T_M      = 80.2 + 273.15               # melting point, K
DH_FUS   = 18804.0                     # J/mol
RHO_20   = 1.0253                      # g/cc at 20 C -- see the note below
# log10 P^sub(bar) = A - B/T, T in K
PSUB_A, PSUB_B = 8.722, 3783.0
# Table 9.6-1, n-hexane
V_HEX, DELTA_HEX = 132.0, 7.3          # cc/mol, (cal/cc)^(1/2)

# WHICH PHASE IS 1.0253 g/cc?  The illustration's data line calls it the density of the
# SOLID, and its solution then argues that the volume change on melting is small so the
# solid's volume may stand in for the liquid's. Neither half of that holds:
#
#   * 1.0253 g/cc at 20 C is an early-20th-century relative density (d_4^20) that
#     entered the CRC Handbook -- the illustration's own cited source -- and from there
#     the safety-data and database literature. Modern crystallographic work puts solid
#     naphthalene at about 1.162 g/cc at 20 C, i.e. 110.3 cc/mol.
#   * the volume change on melting is therefore about 19 %, not "small".
#
# But the NUMBER is right for the job it does here. Regular solution theory needs the
# molar volume of the SUBCOOLED LIQUID -- Eq. 12.1-11 defines delta through it, and the
# volume fraction Phi_2 mixes liquid volumes -- and extrapolating the liquid density
# from the 0.9625 g/cc at 100 C given two lines above gives 125.2 cc/mol at 20 C. So
# 128.19/1.0253 = 125 cc/mol is the subcooled-liquid molar volume, reached by a route
# that describes it as the solid's.
#
# Do not "correct" this to 110.3: that is the solid volume, it would also change
# delta_1 (which is computed FROM this volume), and the pair together move x_1 from
# 0.077 to 0.042 against a measured 0.09. The solid molar volume matters where a solid
# Poynting factor appears -- Illustrations 7.4-9, 12.1-5 and 12.1-6 -- not here.
V_N = MW_N / RHO_20                    # 125.03 cc/mol, the SUBCOOLED LIQUID

print("  Illustration 12.1-1, the ideal solubility")
print(f"    V_1^L (subcooled liquid)     = {V_N:.2f} cc/mol      SIS 125 cc/mol")
x_ideal = float(sle.ideal_solubility(T, T_M, DH_FUS))
print(f"    ln(x_1 gamma_1)              = {sle.ln_x_gamma(T, T_M, DH_FUS):.4f}"
      f"           SIS -1.314")
print(f"    x_1 (gamma_1 = 1)            = {x_ideal:.4f}          SIS 0.269")
print(f"\n    measured x_1 = 0.09, so the ideal answer is high by a factor of"
      f" {x_ideal / 0.09:.1f}")
print("    -- which is SIS's own Comment, 'a factor of 3 too large'.")

# The two candidate volumes, side by side, so the choice above is visible and not
# asserted: the solid volume is what a Poynting factor wants, and it is 12 % smaller.
print(f"\n    for contrast, the SOLID at 20 C (rho = 1.162 g/cc):"
      f" {MW_N / 1.162:.2f} cc/mol")
print(f"    the two differ by {(V_N / (MW_N / 1.162) - 1) * 100:.0f} %, and the"
      f" regular-solution route needs the LIQUID one")


  Illustration 12.1-1, the ideal solubility
    V_1^L (subcooled liquid)     = 125.03 cc/mol      SIS 125 cc/mol
    ln(x_1 gamma_1)              = -1.3144           SIS -1.314
    x_1 (gamma_1 = 1)            = 0.2686          SIS 0.269

    measured x_1 = 0.09, so the ideal answer is high by a factor of 3.0
    -- which is SIS's own Comment, 'a factor of 3 too large'.

    for contrast, the SOLID at 20 C (rho = 1.162 g/cc): 110.32 cc/mol
    the two differ by 13 %, and the regular-solution route needs the LIQUID one



## The awkward step: a solubility parameter for a liquid that does not exist

Regular solution theory (Eq. 12.1-9) needs the solubility parameter of the **solute as a
liquid**,

$$RT\ln\gamma_1 = \underline{V}_1^{\rm L}\,\Phi_2^2\,(\delta_1-\delta_2)^2 ,
\qquad \delta_1 = \sqrt{\frac{\Delta_{\rm vap}\underline{U}_1}{\underline{V}_1^{\rm L}}}$$

but at 20 °C liquid naphthalene does not exist -- it is 60 degrees below its melting
point. Table 9.6-1 lists $\delta$ for $n$-hexane and not for naphthalene, and that is
not an omission: you cannot tabulate a property of a phase that is not stable.

Section 12.1's route around it is three steps, and each one is a real measurement
standing in for the missing one:

1. **The sublimation pressure gives the heat of sublimation.** Clausius-Clapeyron
   (Eq. 7.7-5a) turns the slope of $\log_{10}P^{\rm sub}$ against $1/T$ into
   $\Delta_{\rm sub}H = 2.303\,R\,B$.
2. **Subtract the heat of fusion** to get the heat of vaporization of the *subcooled*
   liquid, Eq. 12.1-12. This is where the hypothetical phase enters: the cycle
   solid $\to$ gas minus solid $\to$ liquid is liquid $\to$ gas, whether or not the
   liquid is stable.
3. **Subtract $RT$** for the internal energy, and divide by the molar volume.

The result is a number for a phase nobody can put in a calorimeter.

In [3]:
DH_SUB = float(sle.heat_of_sublimation(PSUB_B))
DH_VAP = DH_SUB - DH_FUS
delta_N = float(sle.solubility_parameter(DH_SUB, DH_FUS, V_N * CC_PER_MOL, T))
delta_N_cal = delta_N / CAL_CC_HALF

print("  The subcooled-liquid solubility parameter, step by step")
print(f"    dH_sub  = ln(10) R ({PSUB_B:.0f})  = {DH_SUB:8.0f} J/mol   SIS 72 434")
print(f"    dH_vap  = dH_sub - dH_fus     = {DH_VAP:8.0f} J/mol")
print(f"    dU_vap  = dH_vap - RT         = {DH_VAP - R * T:8.0f} J/mol   SIS 51 193")
print(f"    delta_1 = sqrt(dU_vap / V_1)  = {delta_N_cal:8.4f} (cal/cc)^1/2   SIS 9.9")
print(f"\n    n-hexane, Table 9.6-1:          {DELTA_HEX:8.1f} (cal/cc)^1/2")
print(f"    difference:                     {delta_N_cal - DELTA_HEX:8.4f}"
      f"   -- and gamma goes as its SQUARE")
print(f"\n    (SIS writes the Clausius-Clapeyron factor as the rounded 2.303 rather")
print(f"     than ln 10 = {np.log(10):.6f}, which is the whole 9 J/mol difference in dH_sub.)")

# SIS re-rounds delta_1 to 9.9 and V_1 to 125 and then iterates on those, so the
# printed sequence 0.063 -> 0.0746 -> 0.0768 -> 0.0772 is a chain of two-figure
# intermediates. Both chains are run here: the recomputed answer is the honest one,
# and the rounded one is what checks against the page.
CHAINS = [("recomputed", V_N, delta_N_cal),
          ("as printed", 125.0, 9.9)]

print("\n  Illustration 12.1-1, regular solution theory")
print(f"    {'chain':<12} {'V_1':>7} {'delta_1':>8} {'x_1':>8} {'gamma_1':>9}"
      f" {'iters':>6} {'moving':>7}")
results = {}
for label, V1, d1 in CHAINS:
    rs = RegularSolution.from_table_9_6_1([V1, V_HEX], [d1, DELTA_HEX])
    r = sle.solubility(T, T_M, DH_FUS, lambda x: rs.gamma([x, 1 - x], T)[0])
    results[label] = r
    print(f"    {label:<12} {V1:7.2f} {d1:8.4f} {r.x:8.4f} {r.gammas[-1]:9.4f}"
          f" {r.iterations:6d} {r.direction:>7}")
print(f"    {'SIS':<12} {125.0:7.2f} {9.9:8.4f} {0.0772:8.4f}")

spread = ((9.9 - DELTA_HEX) ** 2 / (delta_N_cal - DELTA_HEX) ** 2 - 1) * 100
print(f"\n    The gap is rounding, not physics: rounding delta_1 from"
      f" {delta_N_cal:.4f} to 9.9")
print(f"    moves (delta_1 - delta_2)^2 by {spread:.1f} %, and that lands in an exponent.")
print(f"    Neither chain is wrong; the recomputed one is what the notebook carries")
print(f"    forward, and the printed one is what checks against the page.")

  The subcooled-liquid solubility parameter, step by step
    dH_sub  = ln(10) R (3783)  =    72425 J/mol   SIS 72 434
    dH_vap  = dH_sub - dH_fus     =    53621 J/mol
    dU_vap  = dH_vap - RT         =    51183 J/mol   SIS 51 193
    delta_1 = sqrt(dU_vap / V_1)  =   9.8916 (cal/cc)^1/2   SIS 9.9

    n-hexane, Table 9.6-1:               7.3 (cal/cc)^1/2
    difference:                       2.5916   -- and gamma goes as its SQUARE

    (SIS writes the Clausius-Clapeyron factor as the rounded 2.303 rather
     than ln 10 = 2.302585, which is the whole 9 J/mol difference in dH_sub.)

  Illustration 12.1-1, regular solution theory
    chain            V_1  delta_1      x_1   gamma_1  iters  moving
    recomputed    125.03   9.8916   0.0781    3.4392     15 falling
    as printed    125.00   9.9000   0.0774    3.4729     15 falling
    SIS           125.00   9.9000   0.0772

    The gap is rounding, not physics: rounding delta_1 from 9.8916 to 9.9
    moves (delta_1 - delta_2)^2 by 0.


## The book's own iteration, and why $\Phi_2$ has to be corrected

Regular solution theory needs the solvent **volume fraction** $\Phi_2$, not its mole
fraction, and $\Phi_2$ depends on the answer. Illustration 12.1-1 handles that the way
you would by hand: guess that the solute is dilute, so $\Phi_2 \approx 1$; get $x_1$;
notice that $x_1$ is not small; go back and correct $\Phi_2$.

That is the loop, and the printed sequence is $0.063 \to 0.0746 \to 0.0768 \to 0.0772$.
It converges from below, because the first guess $\Phi_2 = 1$ *overestimates* $\gamma_1$
and therefore underestimates $x_1$. Watching the direction is worth as much as watching
the limit -- a solve that walked the wrong way and landed near the right number is a
bug that agreement hides.

In [4]:
rs_book = RegularSolution.from_table_9_6_1([125.0, V_HEX], [9.9, DELTA_HEX])

print("  Illustration 12.1-1's printed sequence, reproduced")
print("    Phi_2 = 1 first, then corrected at every step")

# x0 = 0 IS the "assume x_1 will be small so Phi_2 = 1" first guess, exactly. The
# iterate history then reads: Phi_2 and gamma_1 are evaluated at step i's INPUT, and
# x_1 is what comes out of it -- so the row has to pair history[i] with history[i+1],
# not with itself.
r = sle.solubility(T, T_M, DH_FUS, lambda x: rs_book.gamma([x, 1 - x], T)[0], x0=0.0)
book_seq = [0.063, 0.0746, 0.0768, 0.0772]

print(f"\n    {'step':>5} {'x_1 in':>8} {'Phi_2':>8} {'ln gamma_1':>11}"
      f" {'x_1 out':>8} {'SIS':>8}")
for i in range(min(5, r.iterations)):
    x_in, x_out, g = r.history[i], r.history[i + 1], r.gammas[i]
    phi2 = float(rs_book.volume_fraction([x_in, 1 - x_in])[1])
    sis = f"{book_seq[i]:.4f}" if i < len(book_seq) else ""
    print(f"    {i:5d} {x_in:8.4f} {phi2:8.4f} {np.log(g):11.3f} {x_out:8.4f} {sis:>8}")
print(f"    {'...':>5}")
print(f"    {'limit':>5} {' ':>8} {' ':>8} {' ':>11} {r.x:8.4f} {'0.0772':>8}")

print(f"\n    {r.iterations} iterations, and the sequence is {r.direction} -- which is the")
print(f"    check that matters: Phi_2 = 1 OVERestimates gamma_1, so the first x_1 is")
print(f"    too small and every correction raises it. A solve that fell to the same")
print(f"    limit would be arriving from the wrong side.")

  Illustration 12.1-1's printed sequence, reproduced
    Phi_2 = 1 first, then corrected at every step

     step   x_1 in    Phi_2  ln gamma_1  x_1 out      SIS
        0   0.0000   1.0000       1.451   0.0630   0.0630
        1   0.0630   0.9402       1.282   0.0745   0.0746
        2   0.0745   0.9291       1.252   0.0768   0.0768
        3   0.0768   0.9270       1.246   0.0772   0.0772
        4   0.0772   0.9266       1.245   0.0773         
      ...
    limit                                 0.0774   0.0772

    14 iterations, and the sequence is rising -- which is the
    check that matters: Phi_2 = 1 OVERestimates gamma_1, so the first x_1 is
    too small and every correction raises it. A solve that fell to the same
    limit would be arriving from the wrong side.



## Illustration 12.1-2: the same equation, $\gamma_1$ from UNIFAC

UNIFAC needs no measurement on this mixture at all -- only the group counts. Naphthalene
is eight aromatic CH and two aromatic C; $n$-hexane is two CH$_3$ and four CH$_2$. That
is the whole input.

**Which UNIFAC, though.** `thermo.UNIFAC` carries two parameter sets, and they are two
different regressions of one idea: the classic 1975 set with temperature-independent
$a_{mn}$, and the Dortmund set with $\Psi = \exp[-(a/T + b + cT)]$ and the $r^{3/4}$
combinatorial term. Aspen Plus calls them UNIFAC and UNIF-DMD. They are not
interchangeable, and rather than assume which one the illustration used, compute both
and let them disagree.

In [5]:

from thermo import UNIFAC

NAPHTHALENE = {9: 8, 10: 2}            # 8 ACH + 2 AC
N_HEXANE    = {1: 2, 2: 4}             # 2 CH3 + 4 CH2

print("  Illustration 12.1-2, both parameter sets, from x_1 = 0.07 as SIS does")
print(f"    {'set':<10} {'gamma_1(0.07)':>14} {'first x_1':>10} {'x_1':>8} {'iters':>6}")
unifac = {}
for kind in ("original", "modified"):
    u = UNIFAC(kind)
    gamma = lambda x, u=u: u.gamma([NAPHTHALENE, N_HEXANE], [x, 1 - x], T)[0]
    r = sle.solubility(T, T_M, DH_FUS, gamma, x0=0.07)
    unifac[kind] = r
    print(f"    {kind:<10} {gamma(0.07):14.4f} {r.history[1]:10.4f} {r.x:8.4f}"
          f" {r.iterations:6d}")
print(f"    {'SIS':<10} {3.2726:14.4f} {0.0821:10.4f} {0.0856:8.4f}")

m = unifac["modified"]
print(f"\n  The Dortmund set reproduces the illustration to four figures:")
print(f"    gamma_1(0.07) = {m.gammas[0]:.4f} against SIS's 3.2726")
print(f"    first step    = {m.history[1]:.4f} against SIS's 0.0821")
print(f"    converged     = {m.x:.4f} against SIS's 0.0856")
print(f"  The classic 1975 set does not, and is not close"
      f" ({unifac['original'].x:.4f}).")
print(f"  So Illustration 12.1-2 was computed with the temperature-dependent")
print(f"  (Dortmund) parameters, which is also the default in this package.")

  Illustration 12.1-2, both parameter sets, from x_1 = 0.07 as SIS does
    set         gamma_1(0.07)  first x_1      x_1  iters
    original           2.3888     0.1125   0.1243     15
    modified           3.2725     0.0821   0.0856     14
    SIS                3.2726     0.0821   0.0856

  The Dortmund set reproduces the illustration to four figures:
    gamma_1(0.07) = 3.2725 against SIS's 3.2726
    first step    = 0.0821 against SIS's 0.0821
    converged     = 0.0856 against SIS's 0.0856
  The classic 1975 set does not, and is not close (0.1243).
  So Illustration 12.1-2 was computed with the temperature-dependent
  (Dortmund) parameters, which is also the default in this package.



## The three answers side by side

This is the paired contrast the two illustrations are built to make, and it only works
because the differences are deliberate: same equation, same solid, same solvent, same
temperature, three different sources for one activity coefficient.

In [6]:
X_MEASURED = 0.09                       # SIS footnote 4

rows = [("ideal solution (gamma = 1)",   x_ideal),
        ("regular solution, recomputed", results["recomputed"].x),
        ("regular solution, as printed", results["as printed"].x),
        ("UNIFAC (Dortmund)",            unifac["modified"].x),
        ("UNIFAC (classic 1975)",        unifac["original"].x)]

print(f"  {'gamma_1 from':<30} {'x_1':>8} {'gamma_1':>9} {'x/x_meas':>9}")
for name, x in rows:
    print(f"  {name:<30} {x:8.4f} {x_ideal / x:9.4f} {x / X_MEASURED:9.2f}")
print(f"  {'measured (SIS footnote 4)':<30} {X_MEASURED:8.4f} {'--':>9} {1.00:9.2f}")

print(f"\n  What the comparison actually shows")
print(f"    The ideal-solution answer is not a little wrong, it is a factor of"
      f" {x_ideal / X_MEASURED:.1f} wrong.")
print(f"    Both real models land within 15 %, and the one that needed NO measurement")
print(f"    on this mixture (UNIFAC) does about as well as the one that needed a")
print(f"    sublimation pressure curve (regular solution).")

print(f"\n    And both real models fail on the SAME side, which is the more useful fact:")
print(f"    regular solution theory is"
      f" {(1 - results['recomputed'].x / X_MEASURED) * 100:.0f} % LOW and UNIFAC"
      f" {(1 - unifac['modified'].x / X_MEASURED) * 100:.0f} % low,")
print(f"    while the ideal answer is {(x_ideal / X_MEASURED - 1) * 100:.0f} % HIGH."
      f" Every model that puts gamma_1 > 1 must")
print(f"    predict less solubility than ideal -- Sec. 9.6 notes that regular solution")
print(f"    theory can only do that -- so a solid whose measured solubility EXCEEDED")
print(f"    its ideal value would be outside the reach of this whole route.")

  gamma_1 from                        x_1   gamma_1  x/x_meas
  ideal solution (gamma = 1)       0.2686    1.0000      2.98
  regular solution, recomputed     0.0781    3.4392      0.87
  regular solution, as printed     0.0774    3.4729      0.86
  UNIFAC (Dortmund)                0.0856    3.1401      0.95
  UNIFAC (classic 1975)            0.1243    2.1607      1.38
  measured (SIS footnote 4)        0.0900        --      1.00

  What the comparison actually shows
    The ideal-solution answer is not a little wrong, it is a factor of 3.0 wrong.
    Both real models land within 15 %, and the one that needed NO measurement
    on this mixture (UNIFAC) does about as well as the one that needed a
    sublimation pressure curve (regular solution).

    And both real models fail on the SAME side, which is the more useful fact:
    regular solution theory is 13 % LOW and UNIFAC 5 % low,
    while the ideal answer is 198 % HIGH. Every model that puts gamma_1 > 1 must
    predict less solubi


## Your turn

1. Illustration 12.1-1's Comment stops at "a factor of 3 too large." Plot the ideal
   solubility of naphthalene against temperature from 0 °C to its melting point. Where
   does the ideal answer stop being wrong, and why must it become exact at $T_m$?
   (Equation 12.1-4 is the reason.)
2. Problem 12.1-1 gives measured naphthalene solubilities in chlorobenzene, benzene,
   toluene and carbon tetrachloride at 20 °C -- 0.256, 0.241, 0.224 and 0.205. Three of
   those four are in Table 9.6-1. Predict them with regular solution theory and with
   UNIFAC, and rank the two models by how well they order the four solvents. Getting the
   *order* right matters more for solvent selection than getting each number right.
3. The solubilities in question 2 are all near 0.25, where $x_1$ is not small at all.
   Check the assumption buried in this notebook that the solvent's own activity
   coefficient never appeared: at $x_1 = 0.25$, is the mixture still one where the
   solute equation alone is enough?
4. Rerun the regular-solution chain with the subcooled-liquid density changed by 1 %.
   How much does $x_1$ move? Remember that $\delta_1$ is computed *from* that volume, so
   both factors in $\underline{V}_1^{\rm L}(\delta_1-\delta_2)^2$ move together. That
   sensitivity is why the printed 9.9 and the recomputed 9.8916 give visibly different
   answers, and it is worth knowing before quoting a solubility to three figures.
7. The data line calls 1.0253 g/cm$^3$ the density of the *solid*, but solid naphthalene
   is about 1.162 g/cm$^3$ at 20 °C. Work out which phase the number belongs to by
   extrapolating the liquid density given at 100 °C down to 20 °C, then say why using the
   true solid volume here would make the prediction worse rather than better. Where in
   Chapter 12 *is* the solid molar volume the right quantity?
5. Illustration 12.1-2's iteration converges from *below* (0.07 $\to$ 0.0821 $\to$
   0.0856) and Illustration 12.1-1's converges from below too, but this notebook's
   default start is the ideal solubility, which converges from *above*. Show that both
   reach the same limit, then find a $\gamma(x)$ model steep enough that successive
   substitution diverges. What would you do instead?
6. Section 12.1 says the same method extends to mixed solvents by replacing Eq. 12.1-8
   with Eq. 12.1-14. Estimate naphthalene's solubility in a 50/50 $n$-hexane + carbon
   tetrachloride mixture, which is Problem 12.1-2, and say whether it lies between the
   two pure-solvent answers.